# Transcripción de audios del panel con Whisper

Este notebook es el paso del medio de `code/transcripciones.qmd`:

1. R descarga los `.ogg` de la ronda y los sube a **`Análisis/A/<ronda>`** en Drive.
2. **Este notebook** lee esa carpeta, transcribe y deja un `.txt` por audio en **`Análisis/B/<ronda>`**.
3. R baja los `.txt` y reemplaza los enlaces en el CSV.

> **Regla que no se puede romper:** el `.txt` tiene que llamarse igual que el `.ogg`
> (`q3_fila57.ogg` → `q3_fila57.txt`). R usa ese nombre para saber a qué fila y a qué
> pregunta corresponde la transcripción.

**Es reanudable:** la celda de transcripción vuelve a mirar qué `.txt` ya existen cada
vez que se ejecuta, así que si se corta a mitad de camino se la puede correr de nuevo
y sigue solo con lo que falta. No hace falta re-ejecutar las celdas anteriores.

**Antes de correr:** Entorno de ejecución → Cambiar tipo de entorno → GPU (T4).

In [ ]:
#@title 1) Instalar dependencias
!pip install -q faster-whisper

import torch
print("GPU disponible:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
#@title 2) Montar Drive
from google.colab import drive
drive.mount('/content/drive')

## Si la carpeta del equipo es compartida

Colab monta **Mi unidad**, no "Compartido conmigo". Si la carpeta `Análisis` vive en
una carpeta que te compartieron, hay que agregarle un **acceso directo** en Mi unidad,
una sola vez: en Drive, click derecho sobre la carpeta `Análisis` → *Organizar* →
*Añadir acceso directo* → Mi unidad. Ojo que tiene que ser el acceso directo de la
**carpeta**, no del notebook.

La celda siguiente busca la carpeta sola y avisa si no la encuentra.

In [ ]:
#@title 3) Encontrar la carpeta de la ronda

ROUND_ID = "R1"          #@param {type:"string"}
NOMBRE_RAIZ = "Análisis" #@param {type:"string"}
RUTA_MANUAL = ""         #@param {type:"string"}
MODELO = "large-v3"      #@param ["large-v3", "medium", "small"]
IDIOMA = "es"            #@param {type:"string"}

import os, glob
from pathlib import Path

def buscar_raiz(nombre):
    """Busca una carpeta llamada `nombre` que contenga las subcarpetas A y B."""
    patrones = [
        f"/content/drive/MyDrive/{nombre}",
        f"/content/drive/MyDrive/*/{nombre}",
        f"/content/drive/MyDrive/*/*/{nombre}",
        f"/content/drive/MyDrive/*/*/*/{nombre}",
        f"/content/drive/.shortcut-targets-by-id/*/{nombre}",
        f"/content/drive/.shortcut-targets-by-id/*/*/{nombre}",
    ]
    candidatos = []
    for p in patrones:
        for c in glob.glob(p):
            if os.path.isdir(os.path.join(c, 'A')) and os.path.isdir(os.path.join(c, 'B')):
                candidatos.append(c)
    return sorted(set(candidatos))

if RUTA_MANUAL:
    raiz = Path(RUTA_MANUAL)
else:
    encontradas = buscar_raiz(NOMBRE_RAIZ)
    if not encontradas:
        raise SystemExit(
            f"No encontré una carpeta '{NOMBRE_RAIZ}' con subcarpetas A y B dentro de Mi unidad.\n"
            "Si la carpeta del equipo es compartida, agregale un acceso directo en Mi unidad\n"
            "(click derecho sobre la CARPETA → Organizar → Añadir acceso directo), o completá RUTA_MANUAL."
        )
    if len(encontradas) > 1:
        print("Hay más de una carpeta candidata; uso la primera:")
        for c in encontradas: print("  -", c)
    raiz = Path(encontradas[0])

dir_audio = raiz / 'A' / ROUND_ID
dir_text  = raiz / 'B' / ROUND_ID
dir_text.mkdir(parents=True, exist_ok=True)

assert dir_audio.exists(), f"No existe {dir_audio}. Subir los audios desde R primero."

def estado():
    """Vuelve a mirar Drive: qué audios hay, qué .txt ya están y qué falta."""
    audios = sorted(dir_audio.glob('*.ogg'))
    hechos = {p.stem for p in dir_text.glob('*.txt')}
    return audios, hechos, [p for p in audios if p.stem not in hechos]

audios, hechos, pendientes = estado()
print("carpeta:", raiz)
print(f"audios: {len(audios)} | ya transcritos: {len(hechos)} | pendientes: {len(pendientes)}")

In [ ]:
#@title 4) Cargar el modelo
from faster_whisper import WhisperModel

compute_type = "float16" if torch.cuda.is_available() else "int8"
device       = "cuda" if torch.cuda.is_available() else "cpu"

model = WhisperModel(MODELO, device=device, compute_type=compute_type)
print(f"modelo {MODELO} cargado en {device} ({compute_type})")

In [ ]:
#@title 5) Transcribir (reanudable: se puede volver a correr esta celda sola)
import time, shutil, traceback
from pathlib import Path

PASADAS = 3   #@param {type:"integer"}

TMP = Path('/content/txt_tmp'); TMP.mkdir(exist_ok=True)
LOG = Path('/content/errores_transcripcion.txt')

def drive_viva():
    """Comprueba que Drive siga montado y escribible."""
    try:
        p = dir_text / '.check'
        p.write_text('ok', encoding='utf-8')
        p.unlink()
        return True
    except Exception:
        return False

def remontar():
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    time.sleep(5)

errores = []
t0 = time.time()

for pasada in range(1, PASADAS + 1):

    audios, hechos, faltan = estado()   # se relee Drive en cada pasada
    if not faltan:
        print(f"no queda nada pendiente ({len(hechos)} de {len(audios)} transcritos)")
        break

    print(f"--- pasada {pasada}: {len(faltan)} pendientes de {len(audios)} ---")

    for i, p in enumerate(faltan, 1):

        destino = dir_text / f"{p.stem}.txt"
        if destino.exists():            # otra pasada ya lo hizo
            continue

        try:
            segmentos, info = model.transcribe(
                str(p),
                language=IDIOMA,
                vad_filter=True,        # recorta silencios, evita alucinaciones
                beam_size=5,
            )
            texto = " ".join(s.text.strip() for s in segmentos).strip()

            # se escribe primero local y después se copia a Drive: si el montaje
            # se corta, no queda un .txt a medio escribir en la carpeta del equipo
            tmp = TMP / f"{p.stem}.txt"
            tmp.write_text(texto, encoding='utf-8')
            shutil.copyfile(tmp, destino)

            print(f"[{i}/{len(faltan)}] {p.name} ({info.duration:.0f}s) -> {texto[:60]!r}")

        except Exception as e:
            errores.append((p.name, traceback.format_exc()))
            print(f"[{i}/{len(faltan)}] ERROR en {p.name}: {type(e).__name__}: {e}")

            # el error mas comun en corridas largas es que se caiga el montaje
            # de Drive: en ese caso se remonta y se sigue con el resto
            if not drive_viva():
                print("   se cayó la conexión con Drive: remonto y sigo")
                remontar()

audios, hechos, faltan = estado()
print(f"\n{len(hechos)} de {len(audios)} transcritos en {(time.time()-t0)/60:.1f} min")
print(f"pendientes: {len(faltan)} | errores registrados: {len(errores)}")

if errores:
    LOG.write_text("\n\n".join(f"### {n}\n{tb}" for n, tb in errores), encoding='utf-8')
    tipos = {}
    for n, tb in errores:
        clave = tb.strip().splitlines()[-1][:120]
        tipos[clave] = tipos.get(clave, 0) + 1
    print("\ntipos de error:")
    for k, v in sorted(tipos.items(), key=lambda x: -x[1]):
        print(f"  {v}x  {k}")
    print(f"\ndetalle completo en {LOG} (descargalo si hay que revisarlo)")
    print("volver a correr esta celda reintenta solo lo que falta")

In [ ]:
#@title 6) Control final
audios, hechos, faltan = estado()
txts = sorted(dir_text.glob('*.txt'))
vacios = [p.name for p in txts if not p.read_text(encoding='utf-8').strip()]

print(f"audios: {len(audios)} | txt generados: {len(txts)}")
print(f"txt vacíos (audio sin voz detectada): {len(vacios)}")
if vacios: print("  ", vacios[:20])
print(f"audios sin transcribir: {len(faltan)}")
if faltan: print("  ", [p.name for p in faltan[:20]])

sobrantes = [p.name for p in txts if p.stem not in {a.stem for a in audios}]
if sobrantes:
    print(f"\n.txt que no corresponden a ningún audio de esta ronda: {len(sobrantes)}")
    print("  ", sobrantes[:20])

if not faltan:
    print("\nListo. Volver a R y correr el bloque 5 de code/transcripciones.qmd")